In [1]:
from google.colab import userdata
from google import genai
api_key=userdata.get('Gemini_api_key')
client=genai.Client(api_key=api_key)

In [2]:
def ask_gemini(prompt):
  response=client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )
  return response.text

In [3]:
user_query="""
My purchase amount is 12000.
Tell me the discount and final payable amount
"""

In [4]:
def planner_agent(query):
  prompt=f"""
  You are a planning agent.
  User Query:
  {query}
  Create a simple plan.
  return only:
  1. required information
  2. action required
  3. expected result
  """
  return ask_gemini(prompt)

In [5]:
plan=planner_agent(user_query)
print(plan)

1. required information: Discount policy (e.g., percentage or fixed amount)
2. action required: User to provide the applicable discount policy for a purchase of 12000.
3. expected result: Calculate the discount amount and the final payable amount.


In [7]:
def discount_calculator(amount):
  if amount<5000:
    discount_rate=0
  elif amount<=10000:
    discount_rate=0.10
  else:
    discount_rate=0.20
  discount=amount*discount_rate
  final_amount=amount-discount
  return {
      "purchase amount":amount,
      "discount rate": discount_rate*100,
      "discount":discount,
      "final amount":final_amount
  }

In [9]:
purchase_amount = 12000
result = discount_calculator(purchase_amount)
print(result)

{'purchase amount': 12000, 'discount rate': 20.0, 'discount': 2400.0, 'final amount': 9600.0}


In [10]:
def critic_agent(query,result):
  prompt=f"""
  You are a verification Agent.
  User Query:
  {query}
  Result:
  {result}
  Verify whether the result is correct or not
  Check:
  1. Is the discount rule correct?
  2. Is the calculation correct?
  3. Is the final amount correct?
  return:
  Status: Pass/Fail
  Reason
  """
  return ask_gemini(prompt)

In [11]:
review=critic_agent(
    user_query,
    result
)

In [12]:
print(review)

Status: Pass
Reason:
1.  **Discount rule correctness:** The result explicitly states a `discount rate` of 20.0%. While the user query did not specify a discount rule, the result provides one and uses it consistently.
2.  **Calculation correctness:**
    *   Discount calculation: `12000 * (20 / 100) = 2400.0`. This matches the `discount: 2400.0` in the result.
    *   Final amount calculation: `12000 - 2400.0 = 9600.0`. This matches the `final amount: 9600.0` in the result.
3.  **Final amount correctness:** The final amount of 9600.0 is correctly calculated based on the purchase amount and the applied discount.

All components of the result are internally consistent and mathematically correct based on the provided purchase amount and the stated discount rate.


In [13]:
def final_agent(query,plan,result,review):
  prompt=f"""
  You are a customer support agent.
  User Query:
  {query}
  Plan:
  {plan}
  Calculation result:
  {result}
  Verification:
  {review}
  give a simple professional final response:
  1. purchase amount
  2. discount percentage
  3. discount amount
  4. final payable amount
  """
  return ask_gemini(prompt)

In [15]:
final_answer=final_agent(
    user_query,
    plan,
    result,
    review
)

In [16]:
print(final_answer)

Thank you for your query!

Based on your purchase amount of **12000**, and applying a **20.0%** discount:

*   **Purchase Amount:** 12000
*   **Discount Percentage:** 20.0%
*   **Discount Amount:** 2400.0
*   **Final Payable Amount:** 9600.0

Let me know if you have any other questions!
